In [ ]:
from astropy.io import fits 
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd 
from typing import Literal

In [ ]:
def find_variable_column_blocks(
        obs_band_combo: np.ndarray,
        col_groups: dict,
        comp_window=1,
        min_offset=.6,
        max_offset=10.0,
        min_cols_frac=0.3,
        offset_type="auto",
        max_gap=10,
        min_block_size=20,
        end_threshold=1.0,
        end_window=2,
) -> np.ndarray:
    """
    Some evenly spaced groups of columns (typically separated by 16, 32, or 48
    pixels in global mode) are simultaneously higher or lower during an
    observation. They do not get flagged using the dark signal observations
    because their on / off flickering is usually 200-800 lines long. Dark
    observations are around 260 lines long.

    All columns in a group simultaneously increase or decrease at the same line
    and are 1-4 DN higher than neighboring columns across all bands (sometimes
    more at higher bands).

    Returns dictionary with all blocks of relatively high or low col behavior.

    Args:
        obs_band_combo: Median across bands of full observation, could be in DN or
            radiance. Must be dark signal subtracted otherwise the results will
            be dominated by dark signal.
        #TODO: is sum or mean better than median? could do band by band but
            #noisier then
        col_groups: Dictionary with lists of bad columns.
        comp_window: Which neighboring columns to compare bad col values to.
        min_offset: Minimum offset with neighboring columns to be considered
            a bump or dip.
        max_offset: Maximum offset with neighboring columns to be considered
            a bump or dip.
        min_cols_frac: What % of cols in the group must fit the criteria for
            bump or dip for it be considered a "block"
        offset_type: Designate if we are looking for drops, bumps or either.
            String.
        max_gap: How many lines can we skip within a block before it's not a
            block anymore? Most useful in a highly variable terrain context.
        min_block_size: Minimum number of lines in a block.
        end_threshold: Change in DN within a column to designate end of block.
            NOT relative to neighboring columns.
        end_window: Area within which the end could occur for all group cols.
    """
    n_lines, n_cols = obs_band_combo.shape

    block_results = {}

    for group_name, indices in col_groups.items():
        # unlikely but in case of empty groups
        if not indices:
            block_results[group_name] = {}
            continue

        # cols within a group are 1 indexed bc that's how they are in ds9
        cols = np.array(indices) - 1

        if isinstance(offset_type, str):
            offset_types = [offset_type] * len(cols)
        else:
            offset_types = offset_type

        # positive = drop (neighbors cols higher than col)
        # negative = bump (neighboring cols lower than col)
        offsets = np.full((n_lines, len(cols)), np.nan)

        for j, c in enumerate(cols):
            lo = max(c - comp_window, 0)
            hi = min(c + comp_window + 1, n_cols)
            neighbor_cols = [k for k in range(lo, hi) if k != c]
            # mean or median? median meaningless if it's two cols
            neighbor_mean = obs_band_combo[:, neighbor_cols].mean(axis=1)
            target_val = obs_band_combo[:, c]
            offsets[:, j] = neighbor_mean - target_val

        # vectorize?
        matches = np.zeros_like(offsets, dtype=bool)
        for j, d in enumerate(offset_types):
            col_offset = offsets[:, j]
            mag = np.abs(col_offset)
            in_range = (mag >= min_offset) & (mag <= max_offset)

            # we do know some cols tend to be higher rather than lower but
            # IDK if that's 100% true all the time
            if d == "drop":
                sign_ok = col_offset > 0
            elif d == "bump":
                sign_ok = col_offset < 0
            elif d == "auto":
                sign_ok = np.ones_like(col_offset, dtype=bool)
            else:
                raise ValueError(
                    f"invalid dir type '{d}'")

            matches[:, j] = in_range & sign_ok

        match_fraction = matches.mean(axis=1)
        line_flags = match_fraction >= min_cols_frac

        flagged_idx = np.where(line_flags)[0]
        if len(flagged_idx) == 0:
            block_results[group_name] = {}
            continue

        magnitude_matrix = np.abs(offsets)

        raw_blocks = []
        block_start = flagged_idx[0]
        prev = flagged_idx[0]
        for idx in flagged_idx[1:]:
            if idx - prev > max_gap:
                raw_blocks.append((block_start, prev))
                block_start = idx
            prev = idx
        raw_blocks.append((block_start, prev))

        # look for where the DN suddenly changes across all cols (within the
        # cols themselves, not relative to neighbors)
        delta = np.abs(
            magnitude_matrix[end_window:, :] - magnitude_matrix[:-end_window,
                                               :])
        cols_jumped_frac = np.mean(delta >= end_threshold, axis=1)
        cols_jumped_frac = np.concatenate(
            [np.zeros(end_window), cols_jumped_frac])
        split_points = set(np.where(cols_jumped_frac >= min_cols_frac)[0])

        final_ranges = []
        for (s, e) in raw_blocks:
            seg_start = s
            for line in range(s + 1, e + 1):
                if line in split_points:
                    final_ranges.append((seg_start, line - 1))
                    seg_start = line
            final_ranges.append((seg_start, e))

        final_ranges = [(s, e) for s, e in final_ranges if
                        (e - s + 1) >= min_block_size]

        group_result = {}
        for i, (s, e) in enumerate(final_ranges):
            seg = offsets[s:e + 1, :]

            mean_per_col = np.nanmean(seg, axis=0)
            median_per_col = np.nanmedian(seg, axis=0)

            # get offset_types per column
            offset_per_col = {}
            for j, idx in enumerate(indices):
                col_vals = seg[:, j]
                col_vals = col_vals[~np.isnan(col_vals)]
                n_drop = np.sum(col_vals > 0)
                n_bump = np.sum(col_vals < 0)
                if len(col_vals) == 0:
                    offset_per_col[idx] = "unknown"
                elif n_drop > 0 and n_bump > 0:
                    offset_per_col[idx] = "mixed"
                elif n_drop > 0:
                    offset_per_col[idx] = "drop"
                else:
                    offset_per_col[idx] = "bump"

            # determine offset_type for the whole block
            # may only need this, not to determine it per col. eventually.
            # although for now it's useful.
            flat_vals = seg.flatten()
            flat_vals = flat_vals[~np.isnan(flat_vals)]
            n_drop_block = np.sum(flat_vals > 0)
            n_bump_block = np.sum(flat_vals < 0)
            if len(flat_vals) == 0:
                offset_block = "unknown"
            elif n_drop_block > 0 and n_bump_block > 0:
                offset_block = "mixed"
            elif n_drop_block > 0:
                offset_block = "drop"
            else:
                offset_block = "bump"

            group_result[i] = {
                "start_line": s,
                "end_line": e,
                "n_lines": e - s + 1,
                "mean_offset_per_col": {idx: mean_per_col[j] for
                                              j, idx in enumerate(indices)},
                "median_offset_per_col": {idx: median_per_col[j]
                                                for j, idx in
                                                enumerate(indices)},
                "offset_type_per_col": offset_per_col,
                "mean_offset_block": np.nanmean(seg),
                "median_offset_block": np.nanmedian(seg),
                "offset_type_block": offset_block,
            }

        block_results[group_name] = group_result

    return block_results


def apply_variable_column_correction(
        obs_image: np.ndarray,
        block_results: dict,
        stat: Literal["median", "mean"] = "median",
        method: Literal["column", "block"] = "column",     
        skip_mixed_type: bool = False,       
) -> np.ndarray:
    """
    Apply the offsets found in find_variable_column_blocks to each bad col 
    block. Same offset is applied to all bands of each col, optional to 
    apply a different offset per col within a block or the same offset to all 
    cols within a block. obs_image edited in place. 

    Args:
        obs_image: Input image, in bands x lines x cols. Could be rdn / rfl or 
            DN but need to change settings for anything not DN. 
        block_results: Dict of block info. 
        stat: Correct the columns using either the mean or median value. 
        method: "column" to correct using column-level stats within the block, or 
            using block level stats, "block". 
        skip_mixed_type: if True, columns/blocks with offset_type_per_col "mixed"
            or "unknown" are not corrected. 
    """

    # correction offset amount stored in dict per block / col group 
    stat_name = f"{stat}_offset_per_col" if method == "column" else f"{stat}_offset_block" 

    for group_name, blocks in block_results.items():
        
        if not blocks:
            print('No bad column blocks identified.')
            continue

        # run fix per block of ID'd bad lines, using one offset value per block 
        # or one offset value per column per block 
        for block_idx, block in blocks.items():
            s = block["start_line"]
            e = block["end_line"] + 1 
            types_per_col = block["offset_type_per_col"]

            offset_vals = block[stat_name]

            if method == "block":
                if np.isnan(offset_vals):
                    continue
                if skip_mixed_type and block["offset_type_block"] in ("mixed", "unknown"):
                    # skipping whole block 
                    continue
                for col_idx in types_per_col.keys():
                    c = col_idx - 1
                    obs_image[:, s:e, c] += offset_vals
                    
            elif method == "column": 
                for col_idx, offset in offset_vals.items():
                    if skip_mixed_type and types_per_col.get(col_idx) in ("mixed", "unknown"):
                        # skipping col 
                        continue
                    if np.isnan(offset):
                        continue
                    c = col_idx - 1
                    obs_image[:,s:e, c] += offset

    return obs_image


def fix_variable_columns(obs_image: np.ndarray, col_groups: dict): 
    """
    Wrapper for finding blocks of lines where multiple columns are simultaneously
    higher or lower in DN across bands (caused by some kind of background electronic 
    effect). 

    Identifies blocks of lines where pre-identified column groups are bad and 
    applies an offset. 

    Args: 
        obs_image: Input image, in bands x lines x cols. Could be rdn / rfl or 
            DN but need to change settings for anything not DN. 
        col_groups: Dictionary with lists of bad columns.
    """
    
    block_results = find_variable_column_blocks(
        np.mean(obs_image[:15, :, :], axis=0), 
        col_groups
    ) 

    if any(block_results.values()):
        # only run fix if there's something to fix 
        obs_image = apply_variable_column_correction(
            obs_image, 
            block_results, 
            stat='median',
            method='column',
        )
    else: 
        print('No bad column blocks identified.') 

    # send image back in detector format 
    return obs_image.transpose(1,0,2)

In [ ]:
col_groups = {
                'group1': [310, 278, 246, 230, 198, 166, 150, 118, 86, 70, 38,
                           6],
                'group2': [294, 262, 214, 182, 134, 102, 54, 22],
            }

# with fits.open('/home/bekah/m3-pipeline-dev/data/moon_dss.fits') as hdul:
#     data = hdul[0].data
#     header = hdul[0].header

with fits.open('/home/bekah/m3-pipeline-dev/notebooks/m3g20090417t193320_dss.fits') as hdul:
    data = hdul[0].data
    header = hdul[0].header


corrected = fix_variable_columns(
        obs_image=data,
        col_groups=col_groups,
    )


# looking at det pov not very helpful 
corrected = corrected.transpose(1,0,2)

In [ ]:
fits.writeto('corrected.fits', corrected, overwrite=True)

In [ ]:
# lines show up most in med of all bands image

In [ ]:
fits.writeto('corrected_med.fits', np.median(corrected, axis=0), overwrite=True)
#fits.writeto('corrected_sum.fits', np.sum(corrected, axis=0), overwrite=True)